# 2. ZuCo paper analysis — GPU stage
Open this **after CPU PREPARATION COMPLETE** in the CPU notebook. Select **T4 GPU** and **Run all**. Mount the same Drive account. It validates the CPU handoff, loads the saved trials and runs permutations, inference and four figures. It does **not download native EEG, remap words or repeat cleaning**.

This paper workflow's GPU work is **permutation testing**, not CNN training. Mixed-model inference still runs on CPU. The saved trial archive remains available for later classifiers; this notebook does not silently run the older CNN experiment.

Output stays in `Thesis/Results/word_sentiment_eeg/zuco_paper_figures_staged/`. Budget: 4 GB total, including temporary permutation files. Those work files are removed after successful figures; the raw/clean accepted trial archive is retained. Both notebooks use the same pinned code revision and method fingerprint. Do not run two analyses into the same folder simultaneously.


## Connect Drive and load the pinned code
Choose the same account for Drive mount and quota authentication. No GitHub token is required.

In [ ]:
from google.colab import drive, auth
from pathlib import Path
import os, sys, subprocess, json
from IPython.display import display, Markdown, Image
import pandas as pd

drive.mount('/content/drive')
THESIS=Path('/content/drive/MyDrive/Thesis')
OUT=THESIS/'Results/word_sentiment_eeg/zuco_paper_figures_staged'
REPO=Path('/content/word-sentiment-eeg-paper-staged')
CODE_REVISION='8ed1628a09c108644e47b2929b08da24855f41f1'
def git(*args):
    subprocess.run(['git','-c','credential.helper=','-c','http.extraheader=',*map(str,args)],check=True,
                   env=dict(os.environ,GIT_TERMINAL_PROMPT='0'))
if not REPO.exists():
    git('clone','--depth','1','https://github.com/parmisbathayan/word-sentiment-eeg.git',REPO)
else:
    if subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
        raise RuntimeError('Temporary checkout contains edits. Preserve them before updating.')
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()!=CODE_REVISION:
    git('-C',REPO,'fetch','--depth','1','origin',CODE_REVISION)
    git('-C',REPO,'checkout','--detach',CODE_REVISION)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO/'requirements-paper.txt')],check=True)
print('Code revision:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())

# A mounted Drive filesystem disk_usage value can reflect the runtime volume;
# use the authenticated Drive quota instead.
auth.authenticate_user()
import google.auth
from google.auth.transport.requests import AuthorizedSession
credentials,_=google.auth.default()
response=AuthorizedSession(credentials).get('https://www.googleapis.com/drive/v3/about',
    params={'fields':'storageQuota,user(emailAddress)'},timeout=30)
response.raise_for_status();quota=response.json()['storageQuota']
print('Quota account:',response.json().get('user',{}).get('emailAddress','unavailable'))
if 'limit' not in quota:
    raise RuntimeError('Drive did not expose a finite storage quota; verify free space before starting.')
FREE_DRIVE_BYTES=max(0,int(quota['limit'])-int(quota.get('usage',0)))
print(f'Account free space: {FREE_DRIVE_BYTES/1e9:.2f} GB. Workflow checks the additional space needed before source reads.')
print('Destination:',OUT)


## Load prepared trials and run analysis

In [ ]:
from datetime import datetime, timezone
sys.path.insert(0,str(REPO/'scripts'))
from colab_progress import run_with_progress
command=[sys.executable,'-u',str(REPO/'scripts/zuco_paper_stages.py'),
         '--thesis-root',str(THESIS),'--phase','analyze',
         '--budget-bytes','4000000000','--free-drive-bytes',str(FREE_DRIVE_BYTES)]
attempt=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
run_with_progress(command,OUT/'status'/f'analyze_{attempt}.json',activity='ZuCo paper analyze: persistent CPU to GPU handoff')


## Figure 1 — Selection and cleaning

In [ ]:
display(Image(filename=str(OUT/'figures/figure_1_selection_cleaning.png')))


## Figure 2 — Raw FRPs

In [ ]:
display(Image(filename=str(OUT/'figures/figure_2_raw_frp.png')))


## Figure 3 — Cleaned FRPs

In [ ]:
display(Image(filename=str(OUT/'figures/figure_3_cleaned_frp.png')))


## Figure 4 — Controls

In [ ]:
display(Image(filename=str(OUT/'figures/figure_4_controls.png')))


## Coverage, exclusions and methods

In [ ]:
display(pd.read_csv(OUT/'analysis/source_coverage.csv'))
display(pd.read_csv(OUT/'analysis/exclusion_counts.csv'))
print((OUT/'analysis/analysis_manifest.json').read_text())
display(Markdown((OUT/'METHOD_DIFFERENCES.md').read_text()))
